# Go RL Agent - Training on Google Colab

This notebook trains a Go-playing RL agent using AlphaZero-style self-play with PPO.

**Recommended Runtime:** GPU (Runtime → Change runtime type → GPU)

**Training Time:**
- Fast mode: ~1-2 hours
- Full mode: ~6-12 hours

## 1. Setup

In [ ]:
# Clone the repository or upload files
# For now, we'll assume files are uploaded to /content/go_rl_agent/

import os
import sys

# Check if running on Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab")
except:
    IN_COLAB = False
    print("Running locally")

# Set working directory
if IN_COLAB:
    os.chdir('/content/go_rl_agent')
    !pwd

# Install dependencies
!pip install torch numpy tqdm -q

print("\n[OK] Setup complete")

## 2. Check GPU Availability

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"[OK] GPU available: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("[Warning] No GPU detected. Training will be slow.")
    print("  Go to Runtime → Change runtime type → GPU")

## 3. Test All Components

In [ ]:
# Run comprehensive test
!python test_all.py

## 4. Training Options

Choose your training mode:

### Option A: Fast Mode (Recommended for Testing)

- 20 iterations
- 20 games per iteration
- 100 MCTS simulations
- Training time: ~1-2 hours

In [ ]:
!python train.py --fast

### Option B: Full Training

- 40 iterations
- 50 games per iteration
- 200 MCTS simulations
- Training time: ~6-12 hours

In [ ]:
!python train.py

### Option C: Custom Configuration

In [ ]:
!python train.py --iterations 30 --games 40 --simulations 150

## 5. Monitor Training Progress

In [ ]:
import json
import matplotlib.pyplot as plt

# Load training history
with open('checkpoints/training_history.json', 'r') as f:
    history = json.load(f)

# Plot losses
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].plot(history['iteration'], history['policy_loss'])
axes[0, 0].set_title('Policy Loss')
axes[0, 0].set_xlabel('Iteration')
axes[0, 0].grid(True)

axes[0, 1].plot(history['iteration'], history['value_loss'])
axes[0, 1].set_title('Value Loss')
axes[0, 1].set_xlabel('Iteration')
axes[0, 1].grid(True)

axes[1, 0].plot(history['iteration'], history['entropy'])
axes[1, 0].set_title('Entropy')
axes[1, 0].set_xlabel('Iteration')
axes[1, 0].grid(True)

axes[1, 1].plot(history['iteration'], history['total_loss'])
axes[1, 1].set_title('Total Loss')
axes[1, 1].set_xlabel('Iteration')
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

# Plot evaluation results
if history['eval_results']:
    iterations = [e['iteration'] for e in history['eval_results']]
    random_wins = [e['results']['random']['win_rate'] for e in history['eval_results']]
    capture_wins = [e['results']['capture']['win_rate'] for e in history['eval_results']]
    liberty_wins = [e['results']['liberty']['win_rate'] for e in history['eval_results']]
    
    plt.figure(figsize=(10, 6))
    plt.plot(iterations, random_wins, marker='o', label='vs Random')
    plt.plot(iterations, capture_wins, marker='s', label='vs Capture Heuristic')
    plt.plot(iterations, liberty_wins, marker='^', label='vs Liberty Heuristic')
    plt.xlabel('Iteration')
    plt.ylabel('Win Rate (%)')
    plt.title('Agent Performance vs Baselines')
    plt.legend()
    plt.grid(True)
    plt.show()

## 6. Play Against the Agent

In [ ]:
import torch
from go_env import GoEnv
from network import create_network
from mcts import MCTS

# Load trained model
network = create_network()
try:
    checkpoint = torch.load('checkpoints/model_iter_final.pt', map_location='cpu', weights_only=False)
except TypeError:
    checkpoint = torch.load('checkpoints/model_iter_final.pt', map_location='cpu')
network.load_state_dict(checkpoint['model_state_dict'])
network.eval()

print("[OK] Model loaded")
print("\nHow to play:")
print("  - Enter action number (0-80) for board positions")
print("  - Enter 81 to pass")
print("  - Board positions: row * 9 + col")
print("  - Example: position (2, 3) = 2 * 9 + 3 = 21")

# Create environment
env = GoEnv(board_size=9)
state = env.reset()

env.render()

# Game loop
done = False
while not done:
    # Agent's move (plays Black)
    if env.current_player == 1:
        print("\nAgent thinking...")
        mcts = MCTS(network, num_simulations=200, temperature=0.1)
        action, _ = mcts.get_action(env, add_noise=False)
        print(f"Agent plays: {action}")
    else:
        # Your move
        legal_mask = env.get_legal_moves_mask()
        legal_actions = [i for i in range(82) if legal_mask[i] > 0.5]
        print(f"\nLegal moves: {legal_actions[:10]}..." if len(legal_actions) > 10 else f"Legal moves: {legal_actions}")
        
        while True:
            try:
                action = int(input("Your move: "))
                if action in legal_actions:
                    break
                else:
                    print("Illegal move! Try again.")
            except:
                print("Invalid input!")
    
    # Execute move
    state, reward, done, info = env.step(action)
    env.render()
    
    if done:
        if reward > 0:
            winner = "Black (Agent)" if env.current_player == -1 else "White (You)"
        else:
            winner = "White (You)" if env.current_player == -1 else "Black (Agent)"
        print(f"\nGame Over! Winner: {winner}")

## 7. Download Trained Model

In [ ]:
if IN_COLAB:
    from google.colab import files
    
    # Download final model
    files.download('checkpoints/model_iter_final.pt')
    
    # Download training history
    files.download('checkpoints/training_history.json')
    
    print("[OK] Files downloaded")
else:
    print("Files are saved in the checkpoints/ directory")

## 8. Additional Evaluation

In [ ]:
from evaluate import evaluate_against_baselines, print_evaluation_report

# Run comprehensive evaluation
print("Running comprehensive evaluation (50 games per baseline)...")
results = evaluate_against_baselines(network, num_games=50, num_simulations=200)
print_evaluation_report(results)